<a href="https://colab.research.google.com/github/David-kini/jeux-piere/blob/main/wav2lip-colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
!git clone https://huggingface.co/camenduru/Wav2Lip

# Install yt_dlp, ffmpeg-python, and librosa. This should install a modern librosa and numpy
!pip install yt_dlp ffmpeg-python librosa

%cd Wav2Lip

# Patch librosa to fix the np.complex AttributeError using Python
import os

LIBROSA_CONSTANTQ_PATH="/usr/local/lib/python3.12/dist-packages/librosa/core/constantq.py"
if os.path.exists(LIBROSA_CONSTANTQ_PATH):
    print(f"Patching {LIBROSA_CONSTANTQ_PATH} using Python...")
    with open(LIBROSA_CONSTANTQ_PATH, 'r') as f:
        content = f.read()

    new_content = content.replace('dtype=np.complex,', 'dtype=np.complex128,')

    with open(LIBROSA_CONSTANTQ_PATH, 'w') as f:
        f.write(new_content)
    print("Patch applied.")
else:
    print(f"Warning: {LIBROSA_CONSTANTQ_PATH} not found. Librosa patch skipped.")


fatal: destination path 'Wav2Lip' already exists and is not an empty directory.
/content/Wav2Lip/Wav2Lip
Patching /usr/local/lib/python3.12/dist-packages/librosa/core/constantq.py using Python...
Patch applied.


In [2]:
# @title AI prompt cell

import ipywidgets as widgets
from IPython.display import display, HTML, Markdown,clear_output
from google.colab import ai

dropdown = widgets.Dropdown(
    options=[],
    layout={'width': 'auto'}
)

def update_model_list(new_options):
    dropdown.options = new_options
update_model_list(ai.list_models())

text_input = widgets.Textarea(
    placeholder='Ask me anything....',
    layout={'width': 'auto', 'height': '100px'},
)

button = widgets.Button(
    description='Submit Text',
    disabled=False,
    tooltip='Click to submit the text',
    icon='check'
)

output_area = widgets.Output(
     layout={'width': 'auto', 'max_height': '300px','overflow_y': 'scroll'}
)

def on_button_clicked(b):
    with output_area:
        output_area.clear_output(wait=False)
        accumulated_content = ""
        for new_chunk in ai.generate_text(prompt=text_input.value, model_name=dropdown.value, stream=True):
            if new_chunk is None:
                continue
            accumulated_content += new_chunk
            clear_output(wait=True)
            display(Markdown(accumulated_content))

button.on_click(on_button_clicked)
vbox = widgets.GridBox([dropdown, text_input, button, output_area])

display(HTML("""
<style>
.widget-dropdown select {
    font-size: 18px;
    font-family: "Arial", sans-serif;
}
.widget-textarea textarea {
    font-size: 18px;
    font-family: "Arial", sans-serif;
}
</style>
"""))
display(vbox)


GridBox(children=(Dropdown(layout=Layout(width='auto'), options=('google/gemini-2.5-flash', 'google/gemini-2.5…

## Generate Talking Face Video with Wav2Lip

First, specify the paths to your input video (containing the face) and audio files. You can upload them directly to Colab or use existing paths.

Now, we'll run the Wav2Lip inference script. This will combine the face video and the audio to create a new video where the face appears to be speaking the audio.

In [7]:
import os

# @title Specify Video and Audio Paths

# You can upload your video and audio files directly to Colab,
# or specify paths to files already in your Colab environment.
# For example, if you uploaded 'my_face_video.mp4' and 'my_speech_audio.wav'
# to the /content/ directory, you would set the paths like this:

FACE_VIDEO_PATH = "/content/my_face_video.mp4" # @param {type:"string"}
AUDIO_PATH = "/content/my_speech_audio.wav" # @param {type:"string"}

# Check if the files exist
if not os.path.exists(FACE_VIDEO_PATH):
  print(f"Error: Face video not found at {FACE_VIDEO_PATH}")
if not os.path.exists(AUDIO_PATH):
  print(f"Error: Audio file not found at {AUDIO_PATH}")

print(f"Using face video: {FACE_VIDEO_PATH}")
print(f"Using audio file: {AUDIO_PATH}")


Error: Face video not found at /content/my_face_video.mp4
Error: Audio file not found at /content/my_speech_audio.wav
Using face video: /content/my_face_video.mp4
Using audio file: /content/my_speech_audio.wav


In [8]:
# Ensure you are in the Wav2Lip directory
%cd /content/Wav2Lip

# Run the inference script
# The output video will be saved in the 'results' directory within Wav2Lip
!python inference.py --checkpoint_path checkpoints/wav2lip_gan.pth --face "{FACE_VIDEO_PATH}" --audio "{AUDIO_PATH}" --outfile results/output_talking_face.mp4


/content/Wav2Lip
Using cuda for inference.
Traceback (most recent call last):
  File "/content/Wav2Lip/inference.py", line 280, in <module>
    main()
  File "/content/Wav2Lip/inference.py", line 183, in main
    raise ValueError('--face argument must be a valid path to video/image file')
ValueError: --face argument must be a valid path to video/image file


Finally, let's display the generated talking face video.

In [9]:
from IPython.display import HTML
from base64 import b64encode
import os

output_video_path = '/content/Wav2Lip/results/output_talking_face.mp4'

if os.path.exists(output_video_path):
    mp4 = open(output_video_path, 'rb').read()
    data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
    display(HTML(f"""
    <video width="50%" height="50%" controls>
          <source src="{data_url}" type="video/mp4">
    </video>"""))
else:
    print(f"Output video not found at {output_video_path}. Please check if the inference step completed successfully.")


Output video not found at /content/Wav2Lip/results/output_talking_face.mp4. Please check if the inference step completed successfully.
